In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from wordcloud import WordCloud
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon', quiet=True)
from bertopic import BERTopic
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Replace with your actual file path
df = pd.read_csv('cleaned_reviews.csv')
print(df.shape)
df.head()

In [ ]:
import pandas as pd

# Define the subsets you want to evaluate
subsets_to_check = {
    'right now' : ["ProductId", "UserId", "Time", "Text"],
    'User + Product': ['UserId', 'ProductId'],
    'User + Product + Cleaned Text': ['UserId', 'ProductId', 'cleaned_text'],
    'Cleaned Text Only': ['cleaned_text'],
    'User + Cleaned Text': ['UserId', 'cleaned_text']
}

# Calculate stats for each subset
stats = []
for name, cols in subsets_to_check.items():
    total_rows = len(df)
    unique_combos = df.drop_duplicates(subset=cols).shape[0]
    rows_to_remove = total_rows - unique_combos
    
    # Frequency analysis
    combo_counts = df[cols].value_counts()
    dup_groups = (combo_counts > 1).sum()  # How many unique combos appear >1 time
    max_copies = combo_counts.max()        # Highest duplication count for a single combo
    
    stats.append({
        'Subset': name,
        'Total Rows': total_rows,
        'Rows to Remove': rows_to_remove,
        '% Data Affected': f"{(rows_to_remove / total_rows) * 100:.2f}%",
        'Duplicate Groups': dup_groups,
        'Max Copies of 1 Group': max_copies
    })

# Display as a clean table
stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

In [ ]:
# Pick a subset to inspect, e.g., 'Cleaned Text Only'
subset = ['cleaned_text', 'UserId']
top_dups = df[subset].value_counts().head(10)

print("Top 10 Most Duplicated Combinations:")
for combo, count in top_dups.items():
    print(f"\n{'-'*40}\nCount: {count}\n{combo}")

In [ ]:
# 🌟 Updated WordCloud with Stopwords Filtering
from wordcloud import STOPWORDS
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

# 1️⃣ Combine built-in WordCloud stopwords + NLTK English stopwords
stop_words = STOPWORDS.union(set(nltk_stopwords.words('english')))

# 2️⃣ Add custom filler words common in Amazon reviews (highly recommended for cleaner output)
stop_words.update([
    # HTML artifacts
    'br', 'href', 'http', 'www', 'html',
    
    # Generic verbs (not informative)
    'dont', 'cant', 'wont', 'didnt', 'doesnt', 'isnt', 'wasnt', 'arent', 'without', 'couldnt', 'shouldnt', 'wouldnt',
    'say', 'said', 'tell', 'told', 'think', 'thought', 'know', 'knew',
    'see', 'saw', 'look', 'looked', 'want', 'wanted', 'need', 'needed',
    'get', 'got', 'gotten', 'make', 'made', 'take', 'took', 'give', 'gave',
    'go', 'went', 'gone', 'come', 'came', 'find', 'found', 'try', 'tried', 'though'
    'use', 'used', 'buy', 'bought'
    
    # Vague quantifiers/adverbs
    'much', 'many', 'little', 'lot', 'lots', 'way', 'still', 'well',
    'first', 'last', 'next', 'new', 'old', 'good', 'great', 'best',
    'better', 'bad', 'worst', 'nice', 'fine', 'pretty', 'really',
    'just', 'also', 'too', 'very', 'so', 'even', 'ever', 'never', 'definitely', 'absolutely', 'totally', 'completely', 'extremely',
    
    # Common review filler
    'product', 'products', 'item', 'items', 'thing', 'things', 'stuff', 'review', 'reviews', 'amazon', 'seller'
    'time', 'times', 'day', 'days', 'week', 'month', 'year', 'years',
    'one', 'two', 'three', 'four', 'five',
    'im', 'ive', 'ive', 'youre', 'theyre', 'weve', 'theres', 'heres',
    'would', 'could', 'should', 'will', 'can', 'may', 'might', 'must',
    'than', 'then', 'now', 'here', 'there', 'where', 'when', 'why', 'how', 'much', 'add'
])

# 3️⃣ Generate WordCloud with stopwords applied and lowercase transformation
all_text = ' '.join(df['cleaned_text'].str.lower().sample(frac=0.5, random_state=42))
wc = WordCloud(
    width=800, 
    height=400, 
    background_color='white', 
    max_words=150,
    stopwords=stop_words,
    collocations=True
).generate(all_text)

plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Reviews (Stopwords Removed)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1️⃣ Basic statistics
print("📈 HelpfulnessNumerator - Basic Stats:")
print(df['HelpfulnessNumerator'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

# 2️⃣ Count of reviews with zero helpfulness votes
zero_votes = (df['HelpfulnessNumerator'] == 0).sum()
print(f"\n🔹 Reviews with 0 helpful votes: {zero_votes:,} ({zero_votes/len(df)*100:.1f}%)")

# 5️⃣ Cumulative Distribution Function (CDF) - MOST USEFUL for threshold selection
plt.figure(figsize=(8, 5))
sorted_votes = np.sort(df['HelpfulnessNumerator'])
cdf = np.arange(1, len(sorted_votes) + 1) / len(sorted_votes)
plt.plot(sorted_votes, cdf, linewidth=2, color='navy')
plt.axvline(x=10, color='red', linestyle='--', label='Threshold = 10')
plt.axvline(x=20, color='green', linestyle='--', label='Threshold = 20')
plt.axvline(x=50, color='orange', linestyle='--', label='Threshold = 50')
plt.title('Cumulative Distribution of HelpfulnessNumerator')
plt.xlabel('HelpfulnessNumerator (Threshold)')
plt.ylabel('Cumulative % of Reviews')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(0, 100)
plt.show()

# 6️⃣ Quick reference table: How many reviews remain at common thresholds?
print("\n📋 Reviews retained at different thresholds:")
thresholds = [0, 2, 5, 10, 15, 20, 30, 50, 100]
for t in thresholds:
    count = (df['HelpfulnessNumerator'] >= t).sum()
    pct = count / len(df) * 100
    print(f"  ≥ {t:3d} votes → {count:7,} reviews ({pct:5.2f}%)")

In [ ]:
HELPFUL_THRESHOLD = 2

df_model = df[df['HelpfulnessNumerator'] >= HELPFUL_THRESHOLD].copy().reset_index(drop=True)

print(f"📉 Dataset size after filtering: {len(df_model):,} reviews ({len(df_model)/len(df)*100:.1f}% of original)")
print(f"📊 Score distribution (check for bias):")
print(df_model['Score'].value_counts(normalize=True).sort_index().round(3))

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words=list(stop_words))


# Optimized for speed & memory on filtered subset
topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2", 
    language="english",
    vectorizer_model=vectorizer_model,   
    min_topic_size=500,                  
    calculate_probabilities=False,
    verbose=False
)

topics, probs = topic_model.fit_transform(df_model['cleaned_text'].tolist())
df_model['topic'] = topics

# Get topic info
topic_info = topic_model.get_topic_info()
print(topic_info)

In [ ]:
topic_model.visualize_topics(top_n_topics=15)

In [ ]:
topic_model.visualize_barchart(top_n_topics=15)

In [ ]:
# 2️⃣ Sentiment Analysis (VADER is fast & accurate for reviews)
sia = SentimentIntensityAnalyzer()
df_model['sentiment_scores'] = df_model['clean_text'].apply(sia.polarity_scores)
df_model['sentiment'] = df_model['sentiment_scores'].apply(
    lambda x: 'positive' if x['compound'] >= 0.05 else ('negative' if x['compound'] <= -0.05 else 'neutral')
)

# Merge topic names & sentiment stats
topic_sentiment = topic_info.copy()
topic_sentiment = topic_sentiment.merge(
    df_model.groupby('topic')['sentiment'].value_counts(normalize=True).unstack(fill_value=0),
    left_on='Topic', right_index=True, how='left'
)

print("\n📊 Sentiment distribution per topic (top 5):")
print(topic_sentiment[['Topic', 'Count', 'Name', 'negative', 'neutral', 'positive']].head(10))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure df_model has 'topic', 'Score', and 'ProductId' columns
# Filter out outlier topic (-1)
df_analysis = df_model[df_model['topic'] != -1].copy()

# 1️⃣ Aggregate metrics per topic
topic_metrics = df_analysis.groupby('topic').agg(
    review_count=('Score', 'count'),
    avg_rating=('Score', 'mean'),
    median_rating=('Score', 'median'),
    std_rating=('Score', 'std'),
    unique_products=('ProductId', 'nunique'),  # ← KEY METRIC
    unique_users=('UserId', 'nunique')          # Bonus: user diversity
).reset_index()

# 2️⃣ Merge with topic names and keywords
topic_info = topic_model.get_topic_info()
topic_metrics = topic_metrics.merge(topic_info[['Topic', 'Name', 'Representation']], left_on='topic', right_on='Topic', how='left')

# Add top keywords for reference
topic_metrics['top_keywords'] = topic_metrics['topic'].apply(
    lambda t: ', '.join([word for word, _ in topic_model.get_topic(t)][:4])
)

print("📈 Topic Metrics Overview (Top 10 by Review Count):")
cols = ['Name', 'Representation', 'review_count', 'unique_users', 'unique_products', 'avg_rating', 'top_keywords']
topic_metrics.sort_values('review_count', ascending=False).head(15)[cols].round(2)

In [ ]:
# Prepare data for plotting
plot_data = topic_metrics[topic_metrics['topic'] != -1].sort_values('avg_rating', ascending=True).head(15)

fig, ax1 = plt.subplots(figsize=(14, 7))

# ── Left axis: Avg Rating bars ──────────────────────────────────────────
bars = ax1.barh(range(len(plot_data)), plot_data['avg_rating'],
                # xerr=plot_data['std_rating'] / np.sqrt(plot_data['review_count']),
                color=plt.cm.RdYlGn((plot_data['avg_rating'] - 1) / 4),
                edgecolor='black', alpha=0.85, height=0.6, label='Avg Rating')

ax1.set_yticks(range(len(plot_data)))
ax1.set_yticklabels([f"{row['Name'][:30]}..." if len(str(row['Name'])) > 30 else row['Name']
                     for _, row in plot_data.iterrows()], fontsize=9)
ax1.set_xlabel('Average Rating (1–5)', color='black')
ax1.set_xlim(1, 5)

# ── Right axis: Unique Product Count dots ───────────────────────────────
ax2 = ax1.twiny()
ax2.scatter(plot_data['unique_products'], range(len(plot_data)),
            color='steelblue', zorder=5, s=80, marker='D', label='Unique Products')

# Connect dots with horizontal lines for readability
for i, val in enumerate(plot_data['unique_products']):
    ax2.plot([0, val], [i, i], color='steelblue', linewidth=0.8, alpha=0.4, linestyle=':')

ax2.set_xlabel('Unique Product Count', color='steelblue')
ax2.tick_params(axis='x', labelcolor='steelblue')
ax2.set_xlim(0, plot_data['unique_products'].max() * 1.2)

# ── Legends ─────────────────────────────────────────────────────────────
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=9)

plt.title('Average Rating & Unique Product Count per Topic (Top 15)', pad=20)
plt.tight_layout()
plt.show()

### Recommendations

1. from the topic modeling, we can see the segmentations for each product categories such as kitchen spices and salts (8), pasta noodle products (2), dog foods and products (6), etc. 
2. From here we can also use the rating distribution and unique product count from each of the topics to know the market sentiment or situation regarding each of the product segments. Wether it has low rating and low product count, it can be an opportunity for us to jump in with a better quality product; Or if it has high avg rating but low unique product counts, it can also be an opportunity because there is already well established demands toward that product category, but we still need to be careful as to approach the current existing product brands. And also if it has high avg rating and also high unique product countsm, then the recommendation would be for best to avoid this kind of market because it already quite saturated, only jump in if we confindence we have a very strong differentiator compare to other existing brands